# **Sequence-module for protease inhibtor prediction**

This notebook screens protein sequences for likely **protease inhibitor (PI)** using two fine-tuned deep learning models:  
- **PIP-BERT** (Rostlab/prot_bert based model) → [MuthuS97/PIP-BERT](https://huggingface.co/MuthuS97/PIP-BERT)  
- **PIPES-M** (facebook/esm2_t30_150M_UR50D based model) → [MuthuS97/PIPES-M](https://huggingface.co/MuthuS97/PIPES-M)

Both models classify sequences as:  
- **Positive (1)** → Potential protease inhibitor (likely to have PI activity)  
- **Negative (0)** → Potential Non-protease inhibitor (unlikely to have PI activity)

### Input File Format – FASTA Requirements
Upload a **single plain-text FASTA or Multi FASTA file** (.fasta extension).

**Strict rules for correct parsing**:
- Each record starts with a header line beginning with `>`
- Header content follows on the **same line** (e.g., `>UniProtID ID`)  
- Sequence uses **one-letter amino acid codes** (A–Y; uppercase MUST; X, B, Z, U, O allowed)  
- No numbers, gaps (`-`), dots (`.`), asterisks (`*`), or non-standard characters in sequences  
- Lines can wrap (60–80 characters recommended)  
- Blank lines between records are ignored  
- Multiple sequences supported (the notebook processes the whole file)

**Common Errors to Avoid**:
- Missing `>` at start  
- Header split across lines  
- Nucleotide sequences (ATCG...) instead of protein  
- Rich-text formatting (save as plain text via Notepad, VS Code, etc.)  
- Non-amino-acid characters (e.g., numbers, symbols)


### How to Run the Pipeline
1. Run all cells in order (or use Runtime → Run all).  
2. When the upload cell appears → click **Choose Files** and select your FASTA (Must be mature protein sequence).  
3. Wait for inference (fast on GPU; longer on CPU or very large files).  
4. Download generated CSVs:  
   - `PIP_BERT_predictions.csv` → per-sequence results from PIP-BERT  
   - `PIPES-M_predictions.csv` → per-sequence results from PIPES-M  
   - `Permissive_predictions.csv` → positive by **at least one** model (Discovery mode)  
   - `Non_Permissive_predictions.csv` → positive by **both** models ( High-confidence mode)  
   - `Screening_Summary_Report.csv` → quick count overview  

**Tips for large files** (>5,000–10,000 sequences): Use Colab Pro / High-RAM runtime, or split FASTA into chunks.

**Interpretation notes**:
- Probability > 0.5 → predicted positive  
- Higher confidence = model more certain (max of prob_class_1 or prob_class_0)  
- Intersection (both models agree) usually more specific; union more sensitive  

In [1]:
# @title 0. Install Required Packages
!pip install --quiet transformers torch huggingface_hub

In [2]:
# @title 1. Initialization
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer, EsmForSequenceClassification
from torch.utils.data import Dataset, DataLoader, TensorDataset
from google.colab import files, drive
from IPython.display import display, HTML

mount_drive = False # @param {type:"boolean"}
MAX_LEN = 250 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}

if mount_drive:
    drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# @title 2. Model Loading
# PIP-BERT
print("Loading PIP-BERT (ProtBERT-based) from MuthuS97/PIP-BERT")
tokenizer_bert = BertTokenizer.from_pretrained("MuthuS97/PIP-BERT")
model_bert = BertForSequenceClassification.from_pretrained("MuthuS97/PIP-BERT").to(device).eval()
# PIPES-M
print("Loading PIPES-M (ESM-based) from MuthuS97/PIP-BERT")
tokenizer_esm = AutoTokenizer.from_pretrained("MuthuS97/PIPES-M")
model_esm = EsmForSequenceClassification.from_pretrained("MuthuS97/PIPES-M").to(device).eval()
print("Both models loaded successfully.")

Loading PIP-BERT (ProtBERT-based) from MuthuS97/PIP-BERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/593M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

Both models loaded successfully.


In [4]:
# @title 3. Upload and Parse File
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded.")

fasta_filename = list(uploaded.keys())[0]

def parse_fasta(content):
    headers, sequences = [], []
    current_seq = []
    for line in content.splitlines():
        line = line.strip()
        if line.startswith(">"):
            if current_seq: sequences.append("".join(current_seq).upper())
            headers.append(line[1:])
            current_seq = []
        else: current_seq.append(line.replace(" ", ""))
    if current_seq: sequences.append("".join(current_seq).upper())
    return pd.DataFrame({"header": headers, "sequence": sequences})

with open(fasta_filename, "r") as f:
    df = parse_fasta(f.read())

print(f"Loaded {len(df)} sequences.")
display(df.head())

Saving EPIC2B_mature.fasta to EPIC2B_mature.fasta
Loaded 1 sequences.


,header,sequence
0,EPIC2B_mature,QLNGYSKKEVTPEDTELLQKAQSNVSAYNSDVTSRICYLKVDSLET...


In [5]:
# @title 4. Run Inference
class ProtBERTDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length):
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        seq = " ".join(list(self.sequences[idx]))
        return self.tokenizer(seq, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')

print("Running PIP-BERT inference...")
bert_loader = DataLoader(ProtBERTDataset(df['sequence'].tolist(), tokenizer_bert, MAX_LEN), batch_size=BATCH_SIZE)
bert_probs = []
with torch.no_grad():
    for batch in bert_loader:
        out = model_bert(input_ids=batch['input_ids'].squeeze(1).to(device), attention_mask=batch['attention_mask'].squeeze(1).to(device))
        bert_probs.extend(F.softmax(out.logits, dim=1)[:, 1].cpu().numpy())

print("Running PIPES-M inference...")
esm_enc = tokenizer_esm(df['sequence'].tolist(), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
esm_loader = DataLoader(TensorDataset(esm_enc['input_ids'], esm_enc['attention_mask']), batch_size=BATCH_SIZE)
esm_probs = []
with torch.no_grad():
    for ids, mask in esm_loader:
        out = model_esm(input_ids=ids.to(device), attention_mask=mask.to(device))
        esm_probs.extend(F.softmax(out.logits, dim=1)[:, 1].cpu().numpy())
df['prob_class_1_BERT'] = bert_probs
df['prob_class_0_BERT'] = 1 - np.array(bert_probs)
df['predicted_class_id_BERT'] = (df['prob_class_1_BERT'] > 0.5).astype(int)
df['confidence_BERT'] = np.maximum(df['prob_class_1_BERT'], df['prob_class_0_BERT'])
df['prob_class_1_ESM'] = esm_probs
df['prob_class_0_ESM'] = 1 - np.array(esm_probs)
df['predicted_class_id_ESM'] = (df['prob_class_1_ESM'] > 0.5).astype(int)
df['confidence_ESM'] = np.maximum(df['prob_class_1_ESM'], df['prob_class_0_ESM'])
label_map = {0: "Negative (Non-PI)", 1: "Positive (Potential PI)"}
df['predicted_class_BERT'] = df['predicted_class_id_BERT'].map(label_map)
df['predicted_class_ESM'] = df['predicted_class_id_ESM'].map(label_map)
permissive = df[(df['predicted_class_id_BERT'] == 1) | (df['predicted_class_id_ESM'] == 1)]
non_permissive = df[(df['predicted_class_id_BERT'] == 1) & (df['predicted_class_id_ESM'] == 1)]

raw_bert_cols = ['header', 'sequence', 'predicted_class_id_BERT', 'predicted_class_BERT', 'confidence_BERT', 'prob_class_1_BERT', 'prob_class_0_BERT']
raw_esm_cols = ['header', 'sequence', 'predicted_class_id_ESM', 'predicted_class_ESM', 'confidence_ESM', 'prob_class_1_ESM', 'prob_class_0_ESM']

outputs = {
    "PIP_BERT_predictions.csv": df[raw_bert_cols],
    "PIPES_M_predictions.csv": df[raw_esm_cols],
    "Permissive_predictions.csv": permissive,
    "Non_Permissive_predictions.csv": non_permissive
}

for filename, data in outputs.items():
    data.to_csv(filename, index=False)
    if mount_drive: data.to_csv(f"/content/drive/MyDrive/{filename}", index=False)
    print(f"Generated: {filename}")

Running PIP-BERT inference...
Running PIPES-M inference...
Generated: PIP_BERT_predictions.csv
Generated: PIPES_M_predictions.csv
Generated: Permissive_predictions.csv
Generated: Non_Permissive_predictions.csv


In [6]:
# @title 5. Summary
only_bert = len(df[(df['predicted_class_id_BERT'] == 1) & (df['predicted_class_id_ESM'] == 0)])
only_esm = len(df[(df['predicted_class_id_BERT'] == 0) & (df['predicted_class_id_ESM'] == 1)])
both = len(non_permissive)

summary_df = pd.DataFrame({
    "Metric": ["Total Sequences", "PIP-BERT inhibitor Hits", "PIPES-M inhibitor Hits", "Permissive (Union) inhibitor Hits", "Non-Permissive (Intersection) inhibitor Hits"],
    "Count": [len(df), len(df[df['predicted_class_id_BERT']==1]), len(df[df['predicted_class_id_ESM']==1]), len(permissive), len(non_permissive)]
})
summary_df.to_csv("Screening_Summary_Report.csv", index=False)

display(HTML("<h3>Screening Results Overview</h3>"))
display(summary_df)



,Metric,Count
0,Total Sequences,1
1,PIP-BERT inhibitor Hits,1
2,PIPES-M inhibitor Hits,1
3,Permissive (Union) inhibitor Hits,1
4,Non-Permissive (Intersection) inhibitor Hits,1
